# Извлечение текста из документов

In [1]:
from dataclasses import dataclass
from pathlib import Path
import json
from typing import Dict, List, Tuple, Any, Literal
import re
import fitz
from tqdm import tqdm

In [ ]:
class PDFReader:
    def __init__(self):
        self._ws = re.compile(r"\s+")
    
    def 
    
    def _norm_text(self, s: str) -> str:
        return self._ws.sub(" ", s).strip()

    


In [ ]:
def _norm_text(s: str) -> str:
    return re.compile(r"\s+").sub(" ", s).strip()


@dataclass
class PdfAnalysis:
    path: str
    pages_total: int
    text_pages: int
    text_pages_ratio: float
    total_text_chars: int
    avg_text_chars_per_page: float
    image_pages: int
    classification: Literal['digital', 'scanned', 'hybrid']


def analyze_pdf(
    pdf_path: Path,
    min_text_chars_per_page: int = 30,
    digital_text_pages_ratio_threshold: float = 0.6,
    scanned_image_pages_ratio_threshold: float = 0.6,
    image_only_page_max_text_chars: int = 10,
) -> PdfAnalysis:

    doc = fitz.open(str(pdf_path))
    pages_total = doc.page_count

    text_pages = 0
    image_pages = 0
    image_heavy_pages = 0
    total_text_chars = 0

    for i in range(pages_total):
        page = doc.load_page(i)

        text = _norm_text(page.get_text("text") or "")
        text_chars = len(text)
        total_text_chars += text_chars

        if text_chars >= min_text_chars_per_page:
            text_pages += 1

        imgs = page.get_images(full=True)
        has_images = len(imgs) > 0
        if has_images:
            image_pages += 1

        if has_images and text_chars <= image_only_page_max_text_chars:
            image_heavy_pages += 1

    text_pages_ratio = (text_pages / pages_total) if pages_total else 0.0
    image_heavy_ratio = (image_heavy_pages / pages_total) if pages_total else 0.0
    avg_chars = (total_text_chars / pages_total) if pages_total else 0.0

    # Классификация
    if text_pages_ratio >= digital_text_pages_ratio_threshold and image_heavy_ratio < 0.3:
        classification = "digital"
    elif text_pages_ratio < 0.2 and image_heavy_ratio >= scanned_image_pages_ratio_threshold:
        classification = "scanned"
    else:
        classification = "hybrid"

    return PdfAnalysis(
        path=str(pdf_path),
        pages_total=pages_total,
        text_pages=text_pages,
        text_pages_ratio=round(text_pages_ratio, 4),
        total_text_chars=total_text_chars,
        avg_text_chars_per_page=round(avg_chars, 2),
        image_pages=image_pages,
        classification=classification,
    )


def analyze_pdfs_folder(
    folder: str | Path,
    recursive: bool = True,
    **analyze_kwargs: Any,
) -> Tuple[List[str], List[str], Dict[str, Any]]:
    folder = Path(folder)
    pdfs = sorted(folder.rglob("*.pdf")) if recursive else sorted(folder.glob("*.pdf"))

    for_ocr: List[str] = []
    for_text_extract: List[str] = []
    rows: List[Dict[str, Any]] = []

    for pdf in tqdm(pdfs):
        try:
            a = analyze_pdf(pdf, **analyze_kwargs)
            rows.append(a.__dict__)

            if a.classification == "digital":
                for_text_extract.append(a.path)
            else:
                # scanned + hybrid -> OCR
                for_ocr.append(a.path)

        except Exception as e:
            rows.append({"path": str(pdf), "error": str(e)})
            # на всякий случай отправим в OCR (там иногда “вытягивается” больше)
            for_ocr.append(str(pdf))

    report = {
        "folder": str(folder),
        "count_total": len(pdfs),
        "count_for_text_extract": len(for_text_extract),
        "count_for_ocr": len(for_ocr),
        "items": rows,
    }
    return for_ocr, for_text_extract, report


In [18]:
data_dir = '../../data/raw_data'
out_dir = Path("../../data/extracted_data")

for_ocr, for_text, report = analyze_pdfs_folder(
    data_dir,
    recursive=True,
    min_text_chars_per_page=30,
    digital_text_pages_ratio_threshold=0.6,
)

100%|██████████| 174/174 [00:42<00:00,  4.05it/s]


In [19]:
for_ocr

[]

In [20]:
for_text

['../../data/raw_data/4293720391.pdf',
 '../../data/raw_data/4293722445.pdf',
 '../../data/raw_data/4293722520.pdf',
 '../../data/raw_data/4293722888.pdf',
 '../../data/raw_data/4293724067.pdf',
 '../../data/raw_data/4293724473.pdf',
 '../../data/raw_data/4293726411.pdf',
 '../../data/raw_data/4293731092.pdf',
 '../../data/raw_data/4293731372.pdf',
 '../../data/raw_data/4293732009.pdf',
 '../../data/raw_data/4293732328.pdf',
 '../../data/raw_data/4293732872.pdf',
 '../../data/raw_data/4293733057.pdf',
 '../../data/raw_data/4293734043.pdf',
 '../../data/raw_data/4293735693.pdf',
 '../../data/raw_data/4293736359.pdf',
 '../../data/raw_data/4293736459.pdf',
 '../../data/raw_data/4293737313.pdf',
 '../../data/raw_data/4293742760.pdf',
 '../../data/raw_data/4293744119.pdf',
 '../../data/raw_data/4293744724.pdf',
 '../../data/raw_data/4293744725.pdf',
 '../../data/raw_data/4293744727.pdf',
 '../../data/raw_data/4293744728.pdf',
 '../../data/raw_data/4293745120.pdf',
 '../../data/raw_data/429

In [21]:
report

{'folder': '../../data/raw_data',
 'count_total': 174,
 'count_for_text_extract': 174,
 'count_for_ocr': 0,
 'items': [{'path': '../../data/raw_data/4293720391.pdf',
   'pages_total': 46,
   'text_pages': 46,
   'text_pages_ratio': 1.0,
   'total_text_chars': 120778,
   'avg_text_chars_per_page': 2625.61,
   'image_pages': 46,
   'classification': 'digital'},
  {'path': '../../data/raw_data/4293722445.pdf',
   'pages_total': 66,
   'text_pages': 66,
   'text_pages_ratio': 1.0,
   'total_text_chars': 199878,
   'avg_text_chars_per_page': 3028.45,
   'image_pages': 66,
   'classification': 'digital'},
  {'path': '../../data/raw_data/4293722520.pdf',
   'pages_total': 49,
   'text_pages': 49,
   'text_pages_ratio': 1.0,
   'total_text_chars': 157141,
   'avg_text_chars_per_page': 3206.96,
   'image_pages': 49,
   'classification': 'digital'},
  {'path': '../../data/raw_data/4293722888.pdf',
   'pages_total': 39,
   'text_pages': 39,
   'text_pages_ratio': 1.0,
   'total_text_chars': 11156

In [ ]:
# from __future__ import annotations

# from pathlib import Path
# from typing import List, Dict
# import fitz  # PyMuPDF


# def extract_text_digital_pdf(pdf_path: str | Path, *, sort: bool = True) -> List[Dict[str, str]]:
#     """
#     Возвращает список:
#       [{"page": "1", "text": "..."},
#        {"page": "2", "text": "..."}]

#     sort=True часто улучшает порядок чтения (top-left -> bottom-right). :contentReference[oaicite:3]{index=3}
#     """
#     pdf_path = Path(pdf_path)
#     doc = fitz.open(str(pdf_path))

#     out: List[Dict[str, str]] = []
#     for i in range(doc.page_count):
#         page = doc.load_page(i)
#         text = page.get_text("text", sort=sort) or ""
#         out.append({"page": str(i + 1), "text": text})
#     return out


# def save_digital_text_as_txt(pdf_path: str | Path, out_txt_path: str | Path, *, sort: bool = True) -> None:
#     pages = extract_text_digital_pdf(pdf_path, sort=sort)
#     out_txt_path = Path(out_txt_path)
#     out_txt_path.parent.mkdir(parents=True, exist_ok=True)

#     with out_txt_path.open("w", encoding="utf-8") as f:
#         for item in pages:
#             f.write(f"\n\n=== PAGE {item['page']} ===\n")
#             f.write(item["text"])
            
            
# # сохранить отчет
# Path("pdf_analysis_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

# # обработать цифровые
# for pdf in for_text:
#     pdf_path = Path(pdf)
#     out_txt = out_dir / (pdf_path.stem + ".txt")
#     save_digital_text_as_txt(pdf_path, out_txt)

